# MuseTalk v1.5 on Kaggle

This notebook follows the official MuseTalk inference flow and is meant for a Kaggle GPU notebook.

Before you run it:
- Turn `Accelerator` to `GPU`
- Turn `Internet` to `ON`
- Add a dataset that contains one face video and one audio file
- Keep your input paths simple if possible, ideally without spaces

What it does:
- installs MuseTalk and its dependencies
- downloads the official MuseTalk 1.5 weights and required auxiliary models
- lets you point to your own video and audio files under `/kaggle/input`
- runs normal inference and previews the output video

This notebook is set up for `video -> dubbed video` inference. Use a video input for the first run.

In [ ]:
from pathlib import Path
import os
import sys

WORKDIR = Path('/kaggle/working')
REPO_DIR = WORKDIR / 'MuseTalk'
RESULTS_DIR = WORKDIR / 'results'
MODELS_DIR = REPO_DIR / 'models'

print('Python:', sys.version)
print('Workdir:', WORKDIR)
print('Repo dir:', REPO_DIR)

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq ffmpeg git git-lfs

if [ ! -d /kaggle/working/MuseTalk ]; then
  git clone https://github.com/TMElyralab/MuseTalk.git /kaggle/working/MuseTalk
else
  echo "MuseTalk repo already exists, skipping clone."
fi

ffmpeg -version | head -n 1

In [ ]:
%%bash
set -e
python -m pip install -q --upgrade pip setuptools wheel
python -m pip install -q --force-reinstall --no-cache-dir torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
python -m pip install -q -r /kaggle/working/MuseTalk/requirements.txt
python -m pip install -q --no-cache-dir -U openmim
mim install mmengine
mim install "mmcv==2.0.1"
mim install "mmdet==3.1.0"
mim install "mmpose==1.1.0"

In [ ]:
import torch
import mmcv
import mmdet
import mmpose

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('MMCV:', mmcv.__version__)
print('MMDet:', mmdet.__version__)
print('MMPose:', mmpose.__version__)

In [ ]:
from huggingface_hub import snapshot_download
import requests
import gdown

MODELS_DIR.mkdir(parents=True, exist_ok=True)
for subdir in [
    'musetalk',
    'musetalkV15',
    'syncnet',
    'dwpose',
    'face-parse-bisent',
    'sd-vae',
    'whisper',
]:
    (MODELS_DIR / subdir).mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id='TMElyralab/MuseTalk',
    local_dir=str(MODELS_DIR),
    allow_patterns=[
        'musetalk/musetalk.json',
        'musetalk/pytorch_model.bin',
        'musetalkV15/musetalk.json',
        'musetalkV15/unet.pth',
    ],
    local_dir_use_symlinks=False,
)

snapshot_download(
    repo_id='stabilityai/sd-vae-ft-mse',
    local_dir=str(MODELS_DIR / 'sd-vae'),
    allow_patterns=['config.json', 'diffusion_pytorch_model.bin'],
    local_dir_use_symlinks=False,
)

snapshot_download(
    repo_id='openai/whisper-tiny',
    local_dir=str(MODELS_DIR / 'whisper'),
    allow_patterns=['config.json', 'pytorch_model.bin', 'preprocessor_config.json'],
    local_dir_use_symlinks=False,
)

snapshot_download(
    repo_id='yzd-v/DWPose',
    local_dir=str(MODELS_DIR / 'dwpose'),
    allow_patterns=['dw-ll_ucoco_384.pth'],
    local_dir_use_symlinks=False,
)

snapshot_download(
    repo_id='ByteDance/LatentSync',
    local_dir=str(MODELS_DIR / 'syncnet'),
    allow_patterns=['latentsync_syncnet.pt'],
    local_dir_use_symlinks=False,
)

face_parse_path = MODELS_DIR / 'face-parse-bisent' / '79999_iter.pth'
if not face_parse_path.exists():
    gdown.download(
        id='154JgKpzCPW82qINcVieuPH3fZ2e0P812',
        output=str(face_parse_path),
        quiet=False,
    )

resnet_path = MODELS_DIR / 'face-parse-bisent' / 'resnet18-5c106cde.pth'
if not resnet_path.exists():
    with requests.get('https://download.pytorch.org/models/resnet18-5c106cde.pth', stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(resnet_path, 'wb') as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)

print('Weights downloaded to:', MODELS_DIR)

## Set your input paths

Point these to files you added through Kaggle `Add data`.

Example:
- `/kaggle/input/my-musetalk-data/person.mp4`
- `/kaggle/input/my-musetalk-data/voice.wav`

For a first run, use a short clip and keep `MODEL_VERSION = 'v15'`.

In [ ]:
MODEL_VERSION = 'v15'  # use 'v15' or 'v1'
VIDEO_PATH = '/kaggle/input/your-dataset-name/input_video.mp4'
AUDIO_PATH = '/kaggle/input/your-dataset-name/input_audio.wav'
OUTPUT_NAME = 'musetalk_output.mp4'
BATCH_SIZE = 4
FPS = 25
USE_FLOAT16 = True
BBOX_SHIFT = -7  # only used for v1

assert Path(VIDEO_PATH).exists(), f'Video not found: {VIDEO_PATH}'
assert Path(AUDIO_PATH).exists(), f'Audio not found: {AUDIO_PATH}'

print('Video:', VIDEO_PATH)
print('Audio:', AUDIO_PATH)

In [ ]:
from omegaconf import OmegaConf

config = {
    'task_0': {
        'video_path': VIDEO_PATH,
        'audio_path': AUDIO_PATH,
        'result_name': OUTPUT_NAME,
    }
}

if MODEL_VERSION == 'v1':
    config['task_0']['bbox_shift'] = BBOX_SHIFT

config_path = REPO_DIR / 'configs' / 'inference' / 'kaggle_test.yaml'
OmegaConf.save(config=OmegaConf.create(config), f=str(config_path))

print(config_path)
print(config_path.read_text())

In [ ]:
import shlex
import subprocess

unet_model_path = REPO_DIR / ('models/musetalkV15/unet.pth' if MODEL_VERSION == 'v15' else 'models/musetalk/pytorch_model.bin')
unet_config = REPO_DIR / ('models/musetalkV15/musetalk.json' if MODEL_VERSION == 'v15' else 'models/musetalk/musetalk.json')

command = [
    sys.executable,
    '-m', 'scripts.inference',
    '--inference_config', str(config_path),
    '--result_dir', str(RESULTS_DIR),
    '--unet_model_path', str(unet_model_path),
    '--unet_config', str(unet_config),
    '--version', MODEL_VERSION,
    '--fps', str(FPS),
    '--batch_size', str(BATCH_SIZE),
    '--ffmpeg_path', '/usr/bin',
    '--saved_coord',
]

if USE_FLOAT16:
    command.append('--use_float16')

print('Running command:')
print(' '.join(shlex.quote(part) for part in command))

subprocess.run(command, cwd=str(REPO_DIR), check=True)

In [ ]:
from IPython.display import Video, display

output_dir = RESULTS_DIR / MODEL_VERSION
output_files = sorted(output_dir.glob('*.mp4'), key=lambda path: path.stat().st_mtime)
assert output_files, f'No mp4 output found in {output_dir}'

final_output = output_files[-1]
print('Output video:', final_output)
display(Video(str(final_output), embed=True))